In [1]:
# Kiểm tra tính nhất quán của dữ liệu ngày mà 
# thời gian đầu tiên được ghi nhận

# Thư viện

In [2]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

# Load các file csv

In [3]:
PATH = Path("../data/raw")

files = {
    "nodes": PATH / "nodes.csv",
    "streets": PATH / "streets.csv",
    "segments": PATH / "segments.csv",
    "status": PATH / "segment_status.csv",
    "train": PATH / "train.csv"
}

In [4]:
nodes_df = pd.read_csv(files["nodes"])
streets_df = pd.read_csv(files["streets"])
segments_df = pd.read_csv(files["segments"])
status_df = pd.read_csv(files["status"])
train_df = pd.read_csv(files["train"])

In [5]:
train_nodes = list(set(train_df["s_node_id"]) | set(train_df["e_node_id"]))

In [6]:
train_streets = set(train_df["street_id"].to_list())

# Load graph

In [7]:
with open("../data/raw/osm_train_2020_07_03.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [8]:
osm_data.keys()

dict_keys(['version', 'generator', 'osm3s', 'elements'])

In [9]:
osm_data["version"]

0.6

## OSM Element

In [10]:
osm_elements = pd.DataFrame(osm_data["elements"])

In [11]:
osm_elements.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367285,10.804994,106.721163,NaN,NaN,NaN
1,node,366368275,10.877718,106.776611,NaN,NaN,NaN
2,node,366368291,10.778778,106.764915,NaN,NaN,NaN
3,node,366368454,10.827255,106.715979,NaN,NaN,NaN
4,node,366368543,10.806701,106.719585,NaN,NaN,NaN


In [12]:
osm_elements["type"].unique()

array(['node', 'way', 'relation'], dtype=object)

## OSM Node

In [13]:
osm_nodes_df = osm_elements[osm_elements["type"] == "node"]

In [14]:
osm_nodes_df.head()

,type,id,lat,lon,tags,nodes,members
0,node,366367285,10.804994,106.721163,NaN,NaN,NaN
1,node,366368275,10.877718,106.776611,NaN,NaN,NaN
2,node,366368291,10.778778,106.764915,NaN,NaN,NaN
3,node,366368454,10.827255,106.715979,NaN,NaN,NaN
4,node,366368543,10.806701,106.719585,NaN,NaN,NaN


## OSM Way

In [15]:
osm_way_df = osm_elements[osm_elements["type"] == "way"]

In [16]:
print(osm_way_df.shape)
osm_way_df.head()

(8394, 7)


,type,id,lat,lon,tags,nodes,members
1953,way,32575820,NaN,NaN,"{'highway': 'tertiary', 'surface': 'asphalt', ...","[366450125, 3376089936, 3376089932, 366439221,...",NaN
1954,way,32575862,NaN,NaN,"{'highway': 'secondary', 'name': 'Đường số 5'}","[366469460, 3792257828, 3792257831, 366463672,...",NaN
1955,way,32575864,NaN,NaN,"{'highway': 'secondary', 'maxspeed': '40', 'na...","[1996655233, 6727805054, 5816921541, 6770923492]",NaN
1956,way,32575869,NaN,NaN,"{'name': 'Lê Văn Thịnh', 'highway': 'residenti...","[366441747, 5738173336, 5738173335, 366443557,...",NaN
1957,way,32575935,NaN,NaN,"{'highway': 'trunk_link', 'surface': 'asphalt'...","[1894608446, 3202462314, 2351843539, 320246231...",NaN


In [17]:
data = []

for _, row in osm_way_df.iterrows():
    way_id = row['id']
    nodes = row['nodes']
    if isinstance(nodes, list) and len(nodes) >= 2:
        for i in range(len(nodes)-1):
            data.append({
                'way_id': way_id,
                's_node_id': nodes[i],      # from
                'e_node_id': nodes[i+1]     # to
            })

osm_edges_df = pd.DataFrame(data)
print(osm_edges_df.head())

     way_id   s_node_id   e_node_id
0  32575820   366450125  3376089936
1  32575820  3376089936  3376089932
2  32575820  3376089932   366439221
3  32575820   366439221   366380944
4  32575820   366380944   366428456


In [18]:
osm_reverse_edges_df = osm_edges_df.rename(columns={
    "s_node_id":"e_node_id", 
    "e_node_id":"s_node_id"
})
osm_undirected_edges_df = pd.concat([osm_edges_df, osm_reverse_edges_df])
print(osm_undirected_edges_df.shape)
osm_undirected_edges_df.head()

(116412, 3)


,way_id,s_node_id,e_node_id
0,32575820,366450125,3376089936
1,32575820,3376089936,3376089932
2,32575820,3376089932,366439221
3,32575820,366439221,366380944
4,32575820,366380944,366428456


In [19]:
osm_way_tags_df = pd.json_normalize(
    osm_way_df["tags"]
        .where(
            osm_way_df["tags"]
                .notna(), 
            other=[{}]
        )
)

osm_way_tags_df.insert(0, "way_id", osm_way_df["id"].to_numpy())
osm_way_tags_df.head()

,way_id,highway,surface,name,name:en,name:zh,maxspeed,bicycle,foot,horse,...,old_name:vi:1871-1897,old_name:vi:1897-1955,old_name:zh:1871-1897,old_name:zh:1897-1955,narrow,information,postal_code,maxspeed:advisory,destination:forward,crossing
0,32575820,tertiary,asphalt,Nguyễn Văn Bá,Nguyen Van Ba Street,阮文伯路,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,32575862,secondary,NaN,Đường số 5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32575864,secondary,asphalt,Đường Châu Văn Liêm,NaN,NaN,40,yes,yes,no,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,32575869,residential,NaN,Lê Văn Thịnh,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,32575935,trunk_link,asphalt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Kiểm tra tính nhất quán

## Node

### Node id

In [20]:
osm_nodes = set(osm_nodes_df["id"])
train_nodes = set(train_df["s_node_id"]) | set(train_df["e_node_id"])

In [21]:
inter_train_osm_nodes = osm_nodes & train_nodes

In [22]:
print(len(osm_nodes))
print(len(train_nodes))
print(len(inter_train_osm_nodes))

55445
11314
10418


### Node location

In [23]:
osm_node_locs = set(osm_nodes_df[["id", "lon", "lat"]].apply(tuple, axis=1))
train_node_locs = set(train_df[["s_node_id", "long_snode", "lat_snode"]].apply(tuple, axis=1)) | \
                    set(train_df[["e_node_id", "long_enode", "lat_enode"]].apply(tuple, axis=1))

In [24]:
inter_osm_train_node_locs = osm_node_locs & train_node_locs
print(len(osm_node_locs))
print(len(train_node_locs))
print(len(inter_osm_train_node_locs))

55445
11314
7705


## Way

### Way id

In [25]:
train_segment_ids = set(train_df["street_id"])
osm_ways_ids = set(osm_way_df["id"])
inter_osm_train_edge = osm_ways_ids & train_segment_ids

print(len(train_segment_ids))
print(len(osm_ways_ids))
print(len(inter_osm_train_edge))

1967
8394
1797


### Way nodes

In [26]:
train_segment_nodes = set(train_df[["street_id", "s_node_id", "e_node_id"]].apply(tuple, axis=1))
osm_way_nodes = set(osm_undirected_edges_df.apply(tuple, axis=1))
inter_osm_train_edges = osm_way_nodes & train_segment_nodes

print(len(train_segment_nodes))
print(len(osm_way_nodes))
print(len(inter_osm_train_edges))

10027
116412
5268
